# ResolveAI: LangSmith evaluation of Pinecone retrieval

This tutorial evaluates the 45 labelled prompts in `outputs/resolveai_golden_data/resolveai_golden_data.csv` against the Pinecone-backed retrieval service. Each prompt has an expected knowledge-article title. The experiment measures whether that article is retrieved in the top three results, its rank, and whether the result actually came from Pinecone.

The final cell is deliberately opt-in. Read and run the setup and preview cells first, then set `RUN_EVALUATION = True` only when you are ready to create a LangSmith dataset and experiment.

## 1. Configure credentials

The notebook loads the project `.env` file. It needs OpenAI and Pinecone credentials to create embeddings and query the vector index, plus a LangSmith API key to upload the dataset and evaluation results. Keep all keys in `.env`; never paste them into notebook cells.

In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

# Allow the notebook to run both from this project folder and from its parent.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'services').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'customer support agent'

if not (PROJECT_ROOT / 'services').exists():
    raise FileNotFoundError('Open eval.ipynb from the customer support agent project folder.')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
# override=True ensures a newly saved .env key replaces any stale key in this kernel.
load_dotenv(PROJECT_ROOT / '.env', override=True)

# A dedicated project keeps evaluation experiments easy to find in LangSmith.
os.environ.setdefault('LANGSMITH_PROJECT', 'resolveai-customer-support')

required_variables = [
    'OPENAI_API_KEY',
    'PINECONE_API_KEY',
    'PINECONE_INDEX',
    'LANGSMITH_API_KEY',
]
missing = [name for name in required_variables if not os.getenv(name)]
if missing:
    raise EnvironmentError('Add these values to .env before running the evaluation: ' + ', '.join(missing))

print('Project root:', PROJECT_ROOT)
print('LangSmith project:', os.environ['LANGSMITH_PROJECT'])

Project root: /Users/unni/Documents/Cohort/TGA_Project/customer support agent
LangSmith project: resolveai-customer-support


## 2. Load and validate the golden data

The CSV may include introductory rows above the real header, depending on how it was exported. The loader finds the `Test case ID` header by name instead of relying on a fixed row number. This check protects the evaluation from accidentally using an incomplete or malformed test set.

In [2]:
import csv

CSV_PATH = PROJECT_ROOT / 'outputs' / 'resolveai_golden_data' / 'resolveai_golden_data.csv'
EXPECTED_COLUMNS = {
    'Test case ID',
    'Customer prompt',
    'Expected intent',
    'Expected specialist',
    'Expected knowledge article',
}

with CSV_PATH.open(encoding='utf-8-sig', newline='') as file:
    csv_rows = list(csv.reader(file))

header_index = next(
    (index for index, row in enumerate(csv_rows) if 'Test case ID' in row),
    None,
)
if header_index is None:
    raise ValueError('Could not find the Test case ID header in the golden-data CSV.')

headers = csv_rows[header_index]
if not EXPECTED_COLUMNS.issubset(headers):
    raise ValueError(f'The golden-data CSV is missing expected columns: {EXPECTED_COLUMNS - set(headers)}')

golden_rows = [
    dict(zip(headers, row))
    for row in csv_rows[header_index + 1:]
    if row and row[0].strip()
]

if not golden_rows:
    raise ValueError('No labelled prompts were found below the CSV header.')
if len(golden_rows) != 45:
    raise ValueError(f'Expected 45 labelled prompts, found {len(golden_rows)}.')

print(f'Loaded {len(golden_rows)} labelled prompts.')
golden_rows[:2]

Loaded 45 labelled prompts.


[{'Test case ID': 'BIL-01',
  'Customer prompt': 'Where can I download my subscription invoice?',
  'Expected intent': 'billing',
  'Expected specialist': 'Billing specialist',
  'Expected knowledge article': 'Find invoices and receipts',
  'Expected application behavior': 'Retrieve invoice guidance and consult Billing.',
  'Test Result': 'PASS'},
 {'Test case ID': 'BIL-02',
  'Customer prompt': 'My subscription renewal failed. What should I do?',
  'Expected intent': 'billing',
  'Expected specialist': 'Billing specialist',
  'Expected knowledge article': 'Failed subscription renewal',
  'Expected application behavior': 'Retrieve renewal policy and consult Billing.',
  'Test Result': 'PASS'}]

## 3. Define the Pinecone retrieval target

A LangSmith target is the function evaluated for each dataset example. This target calls the application’s existing `search()` function, which embeds the prompt and queries Pinecone. It raises an error unless `Pinecone retrieval` is reported, preventing a local fallback from being scored as a Pinecone result.

In [3]:
from langsmith import traceable

from services.retrieval import search

@traceable(name='resolveai.eval.pinecone_retrieval_target', run_type='chain')
def pinecone_retrieval_target(inputs: dict) -> dict:
    """Retrieve the top three approved articles for one golden prompt."""
    prompt = inputs['prompt']
    sources, retrieval_mode = search(prompt, limit=3)

    # A successful evaluation must measure Pinecone, not the offline fallback.
    if retrieval_mode != 'Pinecone retrieval':
        raise RuntimeError(
            f'Pinecone retrieval was required, but the application returned: {retrieval_mode}'
        )

    return {
        'retrieval_mode': retrieval_mode,
        'retrieved_articles': [source['title'] for source in sources],
        'retrieved_scores': [round(float(source['score']), 4) for source in sources],
    }

/Users/unni/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 4. Preview one live Pinecone query

Run this small check before the full experiment. It makes one embedding request and one Pinecone query, then displays the article titles and similarity scores that the evaluators will inspect.

In [4]:
preview_case = golden_rows[0]
preview_result = pinecone_retrieval_target({'prompt': preview_case['Customer prompt']})

print('Prompt:', preview_case['Customer prompt'])
print('Expected article:', preview_case['Expected knowledge article'])
print('Retrieved articles:', preview_result['retrieved_articles'])
print('Similarity scores:', preview_result['retrieved_scores'])

/Users/unni/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Prompt: Where can I download my subscription invoice?
Expected article: Find invoices and receipts
Retrieved articles: ['Find invoices and receipts', 'Failed subscription renewal', 'Privacy and personal-data requests']
Similarity scores: [0.6583, 0.4058, 0.3545]


## 5. Build a LangSmith dataset and retrieval evaluators

The CSV rows become versioned LangSmith examples. Before creating anything, this cell verifies that the configured LangSmith credentials can access the account. The evaluators are deterministic and do not call another model: they check recall at 3, reciprocal rank, and the retrieval mode. Deterministic evaluators make failures simple to audit.

In [5]:
from langsmith import Client
from langsmith.evaluation import run_evaluator

DATASET_NAME = 'resolveai-golden-pinecone-retrieval-v1'

# Personal access tokens (lsv2_pt_) use their default workspace. Service keys
# (lsv2_sk_) may need an explicit workspace when they are organization-scoped.
api_key = os.environ['LANGSMITH_API_KEY']
USE_EXPLICIT_WORKSPACE = True  # Set True only for a service key that needs it.
client_options = {'api_key': api_key}
if os.getenv('LANGSMITH_ENDPOINT'):
    client_options['api_url'] = os.environ['LANGSMITH_ENDPOINT']
if api_key.startswith('lsv2_sk_') or USE_EXPLICIT_WORKSPACE:
    if not os.getenv('LANGSMITH_WORKSPACE_ID'):
        raise EnvironmentError('Set LANGSMITH_WORKSPACE_ID for this organization-scoped service key.')
    client_options['workspace_id'] = os.environ['LANGSMITH_WORKSPACE_ID']
else:
    # Do not let a stale workspace ID override the PAT's default workspace.
    os.environ.pop('LANGSMITH_WORKSPACE_ID', None)
def build_langsmith_client() -> Client:
    """Create the remote client only when the evaluation is about to run."""
    return Client(**client_options)

def verify_langsmith_access(client: Client) -> None:
    """Fail early with a useful fix when LangSmith rejects the credentials."""
    try:
        next(client.list_projects(limit=1), None)
    except Exception as exc:
        message = str(exc)
        if '403' in message or 'Forbidden' in message:
            raise PermissionError(
                'LangSmith returned 403 Forbidden. Replace LANGSMITH_API_KEY in .env with a new, '
                'active personal access token from the same LangSmith account that will hold this '
                'evaluation. Restart the kernel and run from the first cell. For an organization-scoped '
                'service key, set USE_EXPLICIT_WORKSPACE = True and use its matching '
                'LANGSMITH_WORKSPACE_ID. Do not put either secret in this notebook.'
            ) from exc
        raise ConnectionError(
            'Could not connect to LangSmith. Check LANGSMITH_ENDPOINT, network access, and the API key.'
        ) from exc

print('Local dataset examples and evaluators are ready.')
print('LangSmith access will be checked only when RUN_EVALUATION is True.')


dataset_examples = [
    {
        'inputs': {'prompt': row['Customer prompt']},
        'outputs': {
            'expected_knowledge_article': row['Expected knowledge article'],
            'expected_intent': row['Expected intent'],
            'expected_specialist': row['Expected specialist'],
        },
        'metadata': {'test_case_id': row['Test case ID']},
    }
    for row in golden_rows
]

@run_evaluator
def expected_article_retrieved(run, example):
    expected = example.outputs['expected_knowledge_article']
    retrieved = run.outputs['retrieved_articles']
    found = expected in retrieved
    return {
        'key': 'expected_article_retrieved_at_3',
        'score': float(found),
        'comment': f'Expected: {expected}. Retrieved: {retrieved}',
    }

@run_evaluator
def expected_article_reciprocal_rank(run, example):
    expected = example.outputs['expected_knowledge_article']
    retrieved = run.outputs['retrieved_articles']
    rank = retrieved.index(expected) + 1 if expected in retrieved else None
    return {
        'key': 'expected_article_reciprocal_rank',
        'score': 1 / rank if rank else 0.0,
        'comment': f'Expected article rank: {rank or ">3"}',
    }

@run_evaluator
def pinecone_was_used(run, example):
    actual_mode = run.outputs['retrieval_mode']
    return {
        'key': 'pinecone_retrieval_used',
        'score': float(actual_mode == 'Pinecone retrieval'),
        'comment': f'Retrieval mode: {actual_mode}',
    }

Local dataset examples and evaluators are ready.
LangSmith access will be checked only when RUN_EVALUATION is True.


## 6. Run the LangSmith experiment

Set `RUN_EVALUATION` to `True` after reviewing the previous cells. The first run creates the named dataset from the CSV. Later runs reuse the same versioned dataset and create a new experiment, allowing you to compare retrieval quality over time. LangSmith stores the detailed result for each prompt, including failures and retrieval scores.

In [6]:
from langsmith import evaluate

RUN_EVALUATION = True  # Change to True only when ready to upload and run.

if RUN_EVALUATION:
    client = build_langsmith_client()
    verify_langsmith_access(client)
    print('LangSmith access verified for project:', os.environ['LANGSMITH_PROJECT'])

    if not client.has_dataset(dataset_name=DATASET_NAME):
        client.create_dataset(
            DATASET_NAME,
            description='ResolveAI golden prompts for Pinecone retrieval evaluation.',
        )
        client.create_examples(dataset_name=DATASET_NAME, examples=dataset_examples)
        print(f'Created LangSmith dataset with {len(dataset_examples)} examples.')
    else:
        print(f'Reusing existing LangSmith dataset: {DATASET_NAME}')

    results = evaluate(
        pinecone_retrieval_target,
        data=DATASET_NAME,
        evaluators=[
            expected_article_retrieved,
            expected_article_reciprocal_rank,
            pinecone_was_used,
        ],
        experiment_prefix='resolveai-pinecone-retrieval',
        description='Top-3 Pinecone retrieval quality against the ResolveAI golden data.',
        metadata={'dataset_source': 'resolveai_golden_data.csv', 'top_k': 3},
        max_concurrency=4,
        client=client,
        blocking=True,
    )
    print('Experiment complete. Open LangSmith to review aggregate metrics and failing examples.')
    results
else:
    print('No LangSmith dataset or experiment created. Set RUN_EVALUATION = True when ready.')

LangSmith access verified for project: resolveai-customer-support
Reusing existing LangSmith dataset: resolveai-golden-pinecone-retrieval-v1
View the evaluation results for experiment: 'resolveai-pinecone-retrieval-995f3099' at:
https://smith.langchain.com/o/7712b4cb-4223-4a0a-88e2-256ee19694c7/datasets/41549a59-12ca-45ed-ad0c-f7179efc209b/compare?selectedSessions=f1fe6671-c0d3-46b5-adec-0382c57dfd14




45it [1:01:38, 82.19s/it] 

Experiment complete. Open LangSmith to review aggregate metrics and failing examples.


## 7. Interpret the results

In LangSmith, open the experiment created by the final cell. Aim for `pinecone_retrieval_used = 1.0`; anything lower means configuration or connectivity caused a fallback. `expected_article_retrieved_at_3` is recall@3. `expected_article_reciprocal_rank` gives more credit when the expected article appears first (1.0), second (0.5), or third (0.33). Filter failed examples to see the customer prompt, expected title, returned titles, and Pinecone scores.